In [1]:
import micropip
await micropip.install(["openpyxl", "pillow"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill

# ── EDIT THESE ────────────────────────────────────────────────────────────────
FUTURE_FILE    = "future data.xlsx"        # uncorrected
CORRECTED_FILE = "corrected future temp.xlsx"
OUTPUT_FILE    = "temporal_analysis.xlsx"

SHEETS = [
    "sorted mid 2-4.5",
    "sorted long 2-4.5",
    "sorted mid 5-8.5",
    "sorted long 5-8.5",
]
LABELS = [
    "SSP2-4.5 Mid (2040-2050)",
    "SSP2-4.5 Long (2070-2080)",
    "SSP5-8.5 Mid (2040-2050)",
    "SSP5-8.5 Long (2070-2080)",
]
MONTHLY_SHEETS = [
    "Monthly_mid_2-4.5",
    "Monthly_long_2-4.5",
    "Monthly_mid_5-8.5",
    "Monthly_long_5-8.5",
]

MONTH1, MONTH2   = 1, 7
M1_NAME, M2_NAME = "January", "July"
MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
# ─────────────────────────────────────────────────────────────────────────────


# ── HELPERS ───────────────────────────────────────────────────────────────────
def bold_cell(ws, row, col, text, size=11, color=None):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, size=size, color=color or "000000")
    return c

def header_cell(ws, row, col, text):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor="2E4057")
    return c

def fig_to_xl_image(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    plt.close(fig)
    return XLImage(buf)


# ── 1. LOAD UNCORRECTED DATA ──────────────────────────────────────────────────
print("Loading uncorrected future data...")
uncorr_monthly = {}   # label -> DataFrame (index=month 1-12, cols=models)
uncorr_daily   = {}   # label -> DataFrame (index=date, cols=models+Ensemble_Mean)

for sheet, label in zip(SHEETS, LABELS):
    df = pd.read_excel(FUTURE_FILE, sheet_name=sheet, header=2)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.set_index("date")
    model_cols = df.columns.tolist()
    df["Ensemble_Mean"] = df[model_cols].mean(axis=1)
    uncorr_daily[label] = df
    monthly = df.copy()
    monthly["month"] = monthly.index.month
    uncorr_monthly[label] = monthly.groupby("month")[model_cols + ["Ensemble_Mean"]].mean()
    print(f"  {label}: loaded")


# ── 2. LOAD CORRECTED DATA ────────────────────────────────────────────────────
print("Loading corrected data...")
corr_monthly = {}
corr_daily   = {}

for sheet, m_sheet, label in zip(SHEETS, MONTHLY_SHEETS, LABELS):
    # Daily
    df = pd.read_excel(CORRECTED_FILE, sheet_name=sheet)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.set_index("date")
    model_cols = [c for c in df.columns if c != "Ensemble_Mean"]
    if "Ensemble_Mean" not in df.columns:
        df["Ensemble_Mean"] = df[model_cols].mean(axis=1)
    corr_daily[label] = df

    # Monthly — use pre-made sheet if it exists, otherwise calculate
    try:
        mdf = pd.read_excel(CORRECTED_FILE, sheet_name=m_sheet)
        mdf = mdf.set_index("Month")
        corr_monthly[label] = mdf
    except Exception:
        tmp = df.copy()
        tmp["month"] = tmp.index.month
        corr_monthly[label] = tmp.groupby("month")[model_cols].mean()
    print(f"  {label}: loaded")


# ── 3. CREATE WORKBOOK ────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)


# ════════════════════════════════════════════════════════════════════════════
# SHEET 1 — MONTHLY COMPARISON
# Layout per scenario block (stacked vertically):
#   Model | Jan_Uncorr | Jan_Corr | Jan_Diff | (gap) | Jul_Uncorr | Jul_Corr | Jul_Diff
# ════════════════════════════════════════════════════════════════════════════
ws1 = wb.create_sheet("Monthly_Comparison")
bold_cell(ws1, 1, 1,
          "Monthly Mean Comparison: Uncorrected vs Corrected Temperature (°C)",
          size=13)

row = 3
all_diffs = []   # collect all differences for summary

for label in LABELS:
    um = uncorr_monthly[label]
    cm = corr_monthly[label]

    # Scenario heading
    c = ws1.cell(row=row, column=1, value=label)
    c.font = Font(bold=True, size=11, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor="4A90D9")
    row += 1

    # Column headers
    cols_h = ["Model",
              f"{M1_NAME} Uncorr", f"{M1_NAME} Corr", f"{M1_NAME} Diff", "",
              f"{M2_NAME} Uncorr", f"{M2_NAME} Corr", f"{M2_NAME} Diff"]
    for ci, h in enumerate(cols_h, 1):
        header_cell(ws1, row, ci, h)
    row += 1

    # Common models
    common = [m for m in um.columns
              if m in cm.columns and m not in ("Ensemble_Mean",)]

    for model in common:
        u1 = um.at[MONTH1, model] if MONTH1 in um.index else np.nan
        c1 = cm.at[MONTH1, model] if MONTH1 in cm.index else np.nan
        d1 = round(c1 - u1, 4)

        u2 = um.at[MONTH2, model] if MONTH2 in um.index else np.nan
        c2 = cm.at[MONTH2, model] if MONTH2 in cm.index else np.nan
        d2 = round(c2 - u2, 4)

        vals = [model,
                round(u1, 4), round(c1, 4), d1, "",
                round(u2, 4), round(c2, 4), d2]
        for ci, v in enumerate(vals, 1):
            ws1.cell(row=row, column=ci, value=v)

        all_diffs.append((abs(d1), label, M1_NAME, model))
        all_diffs.append((abs(d2), label, M2_NAME, model))
        row += 1

    row += 2   # gap between scenarios

# Summary description
all_diffs.sort(reverse=True)
max_val, max_label, max_month, max_model = all_diffs[0]
row += 1
bold_cell(ws1, row, 1, "Summary:", size=11)
row += 1
desc1 = (
    f"The largest correction was applied to {max_model} in {max_label} "
    f"during {max_month} (difference = {max_val:.4f}°C). "
    f"This means that model had the greatest systematic bias relative to the "
    f"observed ERA5 baseline for that month, which the bias correction adjusted. "
    f"Months and models with larger differences indicate stronger historical "
    f"biases that needed more correction, while smaller differences suggest the "
    f"model was already closer to observed conditions."
)
ws1.cell(row=row, column=1, value=desc1).font = Font(size=10)

ws1.column_dimensions["A"].width = 28
for col in ["B","C","D","E","F","G","H"]:
    ws1.column_dimensions[col].width = 17

print("Sheet 1: Monthly Comparison — done")


# ════════════════════════════════════════════════════════════════════════════
# SHEET 2 — TIME SERIES LINE PLOT
# Ensemble mean monthly temperature: uncorrected vs corrected, SSP2-4.5 mid
# ════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Time_Series_Plot")

ts_label = "SSP2-4.5 Mid (2040-2050)"
um_ts = uncorr_monthly[ts_label]
cm_ts = corr_monthly[ts_label]

# Ensemble mean of monthly means across all models
uncorr_ens = (um_ts["Ensemble_Mean"]
              if "Ensemble_Mean" in um_ts.columns
              else um_ts.mean(axis=1))
corr_ens = cm_ts[[c for c in cm_ts.columns]].mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, 13), uncorr_ens.values,
        marker="o", color="#E74C3C", linewidth=2.2,
        label="Uncorrected", markersize=7)
ax.plot(range(1, 13), corr_ens.values,
        marker="s", color="#2E86AB", linewidth=2.2,
        label="Corrected", markersize=7)
ax.fill_between(range(1, 13),
                uncorr_ens.values, corr_ens.values,
                alpha=0.15, color="#888888", label="Correction applied")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_NAMES, fontsize=10)
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Temperature (°C)", fontsize=11)
ax.set_title(
    f"Monthly Mean Temperature — Uncorrected vs Corrected\n{ts_label}",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()

ws2.cell(row=1, column=1,
         value=(
             "The time series shows ensemble mean monthly temperature before and after "
             "bias correction for SSP2-4.5 (2040–2050). The shaded area represents the "
             "magnitude of correction applied each month — larger shaded areas indicate "
             "months where the GCM ensemble had a greater systematic bias relative to "
             "ERA5 observations, requiring a stronger correction."
         )).font = Font(size=10)
ws2.column_dimensions["A"].width = 120

img2 = fig_to_xl_image(fig)
img2.anchor = "A3"
ws2.add_image(img2)

print("Sheet 2: Time Series Plot — done")


# ════════════════════════════════════════════════════════════════════════════
# SHEET 3 — BOX AND WHISKER PLOT
# Compare 2040-2050 vs 2070-2080 for both SSPs using daily ensemble means
# ════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("Box_Whisker_Plot")

data_box = [
    corr_daily["SSP2-4.5 Mid (2040-2050)"] ["Ensemble_Mean"].dropna().values,
    corr_daily["SSP2-4.5 Long (2070-2080)"]["Ensemble_Mean"].dropna().values,
    corr_daily["SSP5-8.5 Mid (2040-2050)"] ["Ensemble_Mean"].dropna().values,
    corr_daily["SSP5-8.5 Long (2070-2080)"]["Ensemble_Mean"].dropna().values,
]
box_labels = [
    "SSP2-4.5\n2040–2050",
    "SSP2-4.5\n2070–2080",
    "SSP5-8.5\n2040–2050",
    "SSP5-8.5\n2070–2080",
]
box_colors = ["#3498DB", "#1A5276", "#E74C3C", "#7B241C"]

fig3, ax3 = plt.subplots(figsize=(10, 6))
bp = ax3.boxplot(
    data_box, labels=box_labels, patch_artist=True,
    flierprops=dict(marker="o", markerfacecolor="gray",
                    markersize=3, alpha=0.4, linestyle="none"),
    medianprops=dict(color="white", linewidth=2.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
)
for patch, col in zip(bp["boxes"], box_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.82)

ax3.set_ylabel("Temperature (°C)", fontsize=11)
ax3.set_title(
    "Daily Temperature Distribution: Mid-Century vs End-Century\n"
    "(Bias-Corrected Ensemble Mean)",
    fontsize=12, fontweight="bold"
)
ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout()

ws3.cell(row=1, column=1,
         value=(
             "The box plot compares the daily corrected temperature distribution across "
             "both SSP scenarios and time periods. A higher median box in 2070–2080 "
             "relative to 2040–2050 confirms continued warming by end-century. "
             "Wider boxes indicate greater seasonal spread, while outlier dots above "
             "the whiskers represent extreme heat days — their increase from mid to "
             "end-century reflects growing frequency of temperature extremes, "
             "particularly under the high-emissions SSP5-8.5 pathway."
         )).font = Font(size=10)
ws3.column_dimensions["A"].width = 120

img3 = fig_to_xl_image(fig3)
img3.anchor = "A3"
ws3.add_image(img3)

print("Sheet 3: Box & Whisker Plot — done")


# ════════════════════════════════════════════════════════════════════════════
# SHEET 4 — ANNUAL VARIABILITY
# Annual mean temperature per year for each scenario, with summary stats
# ════════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet("Annual_Variability")
bold_cell(ws4, 1, 1,
          "Annual Mean Temperature Variability — Bias-Corrected Ensemble Mean (°C)",
          size=12)

# Column headers
for ci, h in enumerate(["Year"] + LABELS, 1):
    header_cell(ws4, 3, ci, h)

# Calculate annual means per year
annual = {}
for label in LABELS:
    df = corr_daily[label][["Ensemble_Mean"]].copy()
    df["year"] = df.index.year
    annual[label] = df.groupby("year")["Ensemble_Mean"].mean()

all_years = sorted(set().union(*[set(annual[l].index) for l in LABELS]))

row = 4
for year in all_years:
    ws4.cell(row=row, column=1, value=year)
    for ci, label in enumerate(LABELS, 2):
        val = annual[label].get(year, np.nan)
        ws4.cell(row=row, column=ci,
                 value=round(val, 4) if not np.isnan(val) else "")
    row += 1

# Summary statistics block
row += 1
bold_cell(ws4, row, 1, "Inter-Annual Statistics:")
row += 1

stat_rows = [
    ("Mean (°C)",         {l: round(annual[l].mean(), 4) for l in LABELS}),
    ("Std Dev (°C)",      {l: round(annual[l].std(),  4) for l in LABELS}),
    ("Min (°C)",          {l: round(annual[l].min(),  4) for l in LABELS}),
    ("Max (°C)",          {l: round(annual[l].max(),  4) for l in LABELS}),
    ("Warmest Year",      {l: int(annual[l].idxmax())    for l in LABELS}),
    ("Coolest Year",      {l: int(annual[l].idxmin())    for l in LABELS}),
]
for stat_name, stat_vals in stat_rows:
    ws4.cell(row=row, column=1, value=stat_name).font = Font(bold=True)
    for ci, label in enumerate(LABELS, 2):
        ws4.cell(row=row, column=ci, value=stat_vals[label])
    row += 1

# Description
row += 1
stds = {l: annual[l].std() for l in LABELS}
max_var = max(stds, key=stds.get)
min_var = min(stds, key=stds.get)

desc4 = (
    f"The annual variability analysis shows year-to-year fluctuations in "
    f"ensemble mean temperature across both scenarios and time periods. "
    f"{max_var} has the highest inter-annual variability (σ = {stds[max_var]:.4f}°C), "
    f"while {min_var} is the most stable (σ = {stds[min_var]:.4f}°C). "
    f"Higher variability alongside a rising mean temperature indicates "
    f"increasing climate instability — even within a single decade, "
    f"individual years may differ significantly from the decadal average."
)
ws4.cell(row=row, column=1, value=desc4).font = Font(size=10)

ws4.column_dimensions["A"].width = 28
for col in ["B", "C", "D", "E"]:
    ws4.column_dimensions[col].width = 22

print("Sheet 4: Annual Variability — done")


# ── SAVE ─────────────────────────────────────────────────────────────────────
wb.save(OUTPUT_FILE)
print(f"\nAll done! Saved to: {OUTPUT_FILE}")

Loading uncorrected future data...
  SSP2-4.5 Mid (2040-2050): loaded
  SSP2-4.5 Long (2070-2080): loaded
  SSP5-8.5 Mid (2040-2050): loaded
  SSP5-8.5 Long (2070-2080): loaded
Loading corrected data...
  SSP2-4.5 Mid (2040-2050): loaded
  SSP2-4.5 Long (2070-2080): loaded
  SSP5-8.5 Mid (2040-2050): loaded
  SSP5-8.5 Long (2070-2080): loaded
Sheet 1: Monthly Comparison — done
Sheet 2: Time Series Plot — done
Sheet 3: Box & Whisker Plot — done
Sheet 4: Annual Variability — done

All done! Saved to: temporal_analysis.xlsx
